# FraudaZero — 03. Comparação de modelos (A6)

Compara regressão logística (baseline), Árvore de Decisão, Random Forest, AdaBoost, MLP e KNN no protocolo de avaliação do projeto (`notebooks/02_experimentos.ipynb`, seção 6):

| Conjunto | Papel |
|---|---|
| Desenvolvimento (primeiros 80% do `train.csv`) | Stratified 5-fold (os mesmos folds para todos) e 5-fold **temporal** (`TimeSeriesSplit`) |
| Holdout temporal (últimos 20% do `train.csv`) | Comparação dos modelos treinados no desenvolvimento, com IC 95% por bootstrap (1.000 reamostragens) |
| `stream.csv` | **Não é usado aqui** (teste final do campeão, A8) |

**Como reproduzir:** com a stack de pé, `python -m src.train compare` (registra uma run por modelo no experimento `comparacao` do MLflow). Este notebook só **lê** essas runs (métricas, `cv_folds.csv`, `cv_temporal_folds.csv` e `holdout_scores.csv`), então todo número abaixo vem do MLflow.

In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix, precision_recall_curve

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
os.environ.setdefault("DATA_DIR", str(ROOT / "data"))
os.environ["MLFLOW_ENABLE_ARTIFACTS_PROGRESS_BAR"] = "false"
from src.validation import PLOT_STYLE

plt.rcParams.update(PLOT_STYLE)

mlflow.set_tracking_uri(os.environ.get("MLFLOW_TRACKING_URI", "http://localhost:5001"))
FIGS = ROOT / "reports" / "figs"

# Uma cor por família de modelo, na ordem fixa da paleta categórica; variantes compartilham a cor.
FAMILY = {
    "logreg": "Regressão logística",
    "decision-tree": "Árvore de Decisão",
    "random-forest": "Random Forest",
    "adaboost": "AdaBoost",
    "mlp": "MLP",
    "knn": "KNN",
}
COLOR = dict(
    zip(FAMILY, ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300"])
)
ORDER = [
    "logreg",
    "decision-tree",
    "random-forest",
    "adaboost-balanced",
    "adaboost-smote",
    "mlp",
    "mlp-smote",
    "knn-under",
    "knn-under-smote",
]


def family(run):
    return next(f for f in FAMILY if run.startswith(f))

## 1. Resultados registrados no MLflow

In [ ]:
exp = mlflow.get_experiment_by_name("comparacao")
runs = mlflow.search_runs([exp.experiment_id], order_by=["attributes.start_time DESC"])
runs = runs.drop_duplicates("tags.mlflow.runName").set_index("tags.mlflow.runName")
runs = runs.loc[[r for r in ORDER if r in runs.index]]

folds, tfolds, scores = {}, {}, {}
for name, run_id in runs["run_id"].items():
    folds[name] = pd.read_csv(
        mlflow.artifacts.download_artifacts(run_id=run_id, artifact_path="cv_folds.csv")
    )
    tfolds[name] = pd.read_csv(
        mlflow.artifacts.download_artifacts(
            run_id=run_id, artifact_path="cv_temporal_folds.csv"
        )
    )
    scores[name] = pd.read_csv(
        mlflow.artifacts.download_artifacts(
            run_id=run_id, artifact_path="holdout_scores.csv"
        )
    )


def m(c):
    return runs[f"metrics.{c}"]


table = pd.DataFrame(
    {
        "tratamento": runs["params.imbalance"],
        "estratificado": m("cv_valid_pr_auc_mean"),
        "estrat. desvio": m("cv_valid_pr_auc_std"),
        "temporal": m("cv_temporal_valid_pr_auc_mean"),
        "temp. desvio": m("cv_temporal_valid_pr_auc_std"),
        "holdout PR-AUC": m("pr_auc"),
        "IC 95% baixo": m("pr_auc_ci_low"),
        "IC 95% alto": m("pr_auc_ci_high"),
        "holdout ROC-AUC": m("roc_auc"),
        "recall@0,5": m("recall"),
        "precisão@0,5": m("precision"),
        "F1@0,5": m("f1"),
        "treino (s)": m("fit_seconds"),
        "predição (ms / mil tx)": m("predict_ms_per_1k"),
    }
)
print("Runs:", dict(runs["run_id"].str[:8]))
table.round(4)

**Leitura da tabela.**
- **PR-AUC no holdout:** a Random Forest lidera (0,759), seguida da regressão logística (0,733), do MLP (0,719) e do KNN (0,716). O intervalo de confiança de 95% de cada modelo tem ~±0,13 de largura (52 fraudes): os quatro primeiros se sobrepõem quase por inteiro (seção 4).
- **Recall, precisão e F1 no limiar 0,5** dependem da escala dos scores de cada modelo, não só da qualidade do ranking. A logística e o AdaBoost balanceado, treinados com pesos de classe, empurram os scores para cima: recall de ~0,90 com precisão de ~0,02 (milhares de falsos alarmes). A RF fica no extremo oposto (precisão 0,95, recall 0,67). Por isso a comparação usa a PR-AUC, que independe do limiar; o limiar é escolhido depois, por modelo (A8).
- **ROC-AUC não ordena como a PR-AUC:** a logística e o AdaBoost com SMOTE têm as maiores ROC-AUC (0,974), mas não a maior PR-AUC. A ROC-AUC é dominada pelas ~45 mil legítimas e pouco sente os falsos alarmes.
- **Custo:** o AdaBoost com SMOTE é o mais caro para treinar (~199 s no desenvolvimento, 13 vezes a RF), e o KNN, o mais caro para pontuar (~27 ms por mil transações, 14 vezes a RF), porque compara cada transação com todo o treino. A logística é a mais barata nas duas pontas.

## 2. Tratamento do desbalanceamento, por modelo

Regra: `class_weight="balanced"` onde o modelo aceita (logística, árvore, Random Forest). Onde não aceita diretamente (AdaBoost, MLP, KNN), testamos SMOTE **dentro do `imblearn.Pipeline`** (só no treino de cada fold) contra uma alternativa: no MLP, sem tratamento; no KNN, só a subamostra; no AdaBoost, o stump com `class_weight="balanced"` como estimador base. A escolha da variante usa **só o desenvolvimento** (escolher pelo holdout gastaria o holdout numa decisão de hiperparâmetro) e o **5-fold temporal**, que é o que estima o desempenho futuro (seção 4).

In [ ]:
pairs = [
    ("adaboost-balanced", "adaboost-smote"),
    ("mlp", "mlp-smote"),
    ("knn-under", "knn-under-smote"),
]
rows = []
for a, b in pairs:
    if a not in folds or b not in folds:
        continue
    d = (
        tfolds[b]["valid_pr_auc"] - tfolds[a]["valid_pr_auc"]
    )  # mesmos splits: diferença pareada
    rows.append(
        {
            "alternativa": a,
            "com SMOTE": b,
            "estrat. alt.": folds[a]["valid_pr_auc"].mean(),
            "estrat. SMOTE": folds[b]["valid_pr_auc"].mean(),
            "temporal alt.": tfolds[a]["valid_pr_auc"].mean(),
            "temporal SMOTE": tfolds[b]["valid_pr_auc"].mean(),
            "splits temporais em que SMOTE ganha": f"{int((d > 0).sum())} de {len(d)}",
            "treino alt. (s)": m("fit_seconds")[a],
            "treino SMOTE (s)": m("fit_seconds")[b],
        }
    )
pd.DataFrame(rows).round(4)

**Leitura.**
- **AdaBoost:** o stump com `class_weight="balanced"` **não funciona dentro do AdaBoost**. O stump balanceado marca fraude em excesso; medido com os pesos do próprio AdaBoost, o erro dele passa de 0,5 já na 4ª rodada, e o algoritmo (SAMME) encerra o boosting com só 3 estimadores. Resultado: PR-AUC de 0,39 (temporal) e 0,10 (holdout). O AdaBoost com SMOTE vence nos 5 splits temporais (0,771). **Escolha: SMOTE.** A primeira tentativa, sem nenhum tratamento, foi descartada por custo: ~1.000 s por treino, contra 137 s com SMOTE (provavelmente pesos que encolhem até virar números subnormais, cuja aritmética é muito lenta).
- **MLP:** pelo estratificado, o SMOTE parece ajudar (0,850 contra 0,843), mas pelo temporal ele atrapalha (0,731 contra 0,755) e só vence em 2 dos 5 splits. As fraudes sintéticas interpolam fraudes do mesmo período; isso ajuda quando a validação tem fraudes "irmãs" (estratificado) e não ajuda no futuro. **Escolha: sem SMOTE.** Na A7, a proporção do SMOTE entra no espaço de busca (de 0,01, quase nada, até 0,5).
- **KNN:** o SMOTE depois da subamostra piora em todos os splits temporais (0,645 contra 0,709). **Escolha: só a subamostra 1:50.**
- Em resumo, o SMOTE só compensou onde o modelo não tinha outro jeito de enxergar a classe rara (AdaBoost).

## 3. PR-AUC no 5-fold estratificado: média, desvio e holdout

In [ ]:
names = list(runs.index)
mean, std, hold = m("cv_valid_pr_auc_mean"), m("cv_valid_pr_auc_std"), m("pr_auc")
fig, ax = plt.subplots(figsize=(7.5, 4.6))
ypos = np.arange(len(names))[::-1]
for y_, n in zip(ypos, names):
    smote = "smote" in n
    ax.barh(
        y_,
        mean[n],
        color=COLOR[family(n)],
        height=0.62,
        alpha=0.55 if smote else 1.0,
        hatch="///" if smote else None,
        edgecolor="#fcfcfb",
        linewidth=0,
    )
    ax.errorbar(mean[n], y_, xerr=std[n], color="#52514e", capsize=3, lw=1.2)
    ax.plot(
        hold[n],
        y_,
        marker="D",
        ms=7,
        color="#0b0b0b",
        mec="#fcfcfb",
        mew=1.5,
        ls="none",
    )
    ax.annotate(
        f"{mean[n]:.3f} ± {std[n]:.3f}",
        (0.01, y_),
        va="center",
        fontsize=8,
        color="#0b0b0b",
    )
ax.plot([], [], marker="D", ms=7, color="#0b0b0b", ls="none", label="Holdout temporal")
ax.errorbar(
    [],
    [],
    xerr=[],
    color="#52514e",
    capsize=3,
    label="Média ± desvio (5-fold estratificado)",
)
ax.set_yticks(ypos, names)
ax.set_xlim(0.0, 1.0)
ax.set_xlabel("PR-AUC (classe fraude)")
ax.set_title("Comparação de modelos — PR-AUC")
ax.grid(axis="y", visible=False)
ax.legend(
    frameon=False, fontsize=8, ncol=2, loc="upper center", bbox_to_anchor=(0.5, -0.12)
)
fig.savefig(FIGS / "cmp_01_pr_auc.png")

**Leitura.** Pelo 5-fold estratificado, a ordem seria RF (0,854) > MLP com SMOTE (0,850) > MLP (0,843) > AdaBoost com SMOTE (0,801), com desvios pequenos (~0,01–0,02), o que sugere diferenças confiáveis. **Mas os losangos (holdout) ficam abaixo das barras em todos os modelos flexíveis**, de 0,04 (KNN) a 0,14 (MLP com SMOTE). Só a logística tem o holdout dentro da barra. A seção seguinte mostra que essa distância vem do embaralhamento do k-fold, não de azar no holdout.

## 4. Estratificado × temporal × holdout

O stratified k-fold embaralha as linhas: transações de momentos diferentes caem no treino e na validação. O 5-fold temporal (`TimeSeriesSplit`) treina sempre no passado e valida no bloco seguinte, como o holdout. Se os modelos flexíveis estiverem aproveitando fraudes próximas no tempo (rajadas), o estratificado fica acima dos outros dois.

In [ ]:
tmean, tstd = m("cv_temporal_valid_pr_auc_mean"), m("cv_temporal_valid_pr_auc_std")
lo, hi = m("pr_auc_ci_low"), m("pr_auc_ci_high")
fig, ax = plt.subplots(figsize=(7.5, 5))
for y_, n in zip(ypos, names):
    ax.errorbar(
        mean[n],
        y_ + 0.22,
        xerr=std[n],
        fmt="o",
        color="#2a78d6",
        ms=6,
        capsize=3,
        lw=1.2,
    )
    ax.errorbar(
        tmean[n], y_, xerr=tstd[n], fmt="s", color="#eb6834", ms=6, capsize=3, lw=1.2
    )
    ax.errorbar(
        hold[n],
        y_ - 0.22,
        xerr=[[hold[n] - lo[n]], [hi[n] - hold[n]]],
        fmt="D",
        color="#0b0b0b",
        ms=6,
        capsize=3,
        lw=1.2,
    )
ax.errorbar(
    [],
    [],
    xerr=[],
    fmt="o",
    color="#2a78d6",
    label="5-fold estratificado (média ± desvio)",
)
ax.errorbar(
    [], [], xerr=[], fmt="s", color="#eb6834", label="5-fold temporal (média ± desvio)"
)
ax.errorbar(
    [],
    [],
    xerr=[],
    fmt="D",
    color="#0b0b0b",
    label="Holdout temporal (IC 95%, bootstrap)",
)
ax.set_yticks(ypos, names)
ax.set_xlim(0.0, 1.0)
ax.set_xlabel("PR-AUC (classe fraude)")
ax.set_title("Três estimativas da PR-AUC por modelo")
ax.grid(axis="y", visible=False)
ax.legend(frameon=False, fontsize=8, loc="lower left")
fig.savefig(FIGS / "cmp_07_estratificado_temporal_holdout.png")

three = pd.DataFrame({"estratificado": mean, "temporal": tmean, "holdout": hold})
three["estrat. − temporal"] = three["estratificado"] - three["temporal"]
three["temporal − holdout"] = three["temporal"] - three["holdout"]
three["treino (estrat.)"] = m("cv_train_pr_auc_mean")
three.round(4)

**Leitura.** Este é o gráfico central da comparação.
- **O estratificado (azul) superestima todos os modelos flexíveis:** fica de 0,03 (AdaBoost com SMOTE) a 0,12 (MLP com SMOTE) acima do temporal (laranja). Na logística, as três estimativas coincidem (0,720 / 0,716 / 0,733).
- **O temporal e o holdout concordam:** a diferença entre os dois é de no máximo ~0,04 em quase todos os modelos (exceções: AdaBoost com SMOTE, 0,075, e o AdaBoost balanceado, que degenerou). Os dois são cortes no tempo; o estratificado é o único que mistura passado e futuro.
- **Mecanismo:** RF e MLP chegam a PR-AUC de **1,0 no treino**, ou seja, memorizam as fraudes. As fraudes vêm em rajadas próximas no tempo (EDA, Fig. 3). No k-fold embaralhado, fraudes de uma mesma rajada caem dos dois lados do corte, e a memorização vira acerto na validação. No corte temporal, as fraudes da validação são de outro momento, e essa vantagem some. A logística não consegue memorizar (treino 0,72) e por isso não ganha nada com o embaralhamento. O AdaBoost de stumps memoriza pouco (treino 0,81) e é o modelo flexível com a menor diferença entre estratificado e temporal.
- **Consequência:** o 5-fold estratificado serve para medir estabilidade, não para estimar o desempenho em produção nem para ordenar modelos de capacidades diferentes. A escolha de variantes (seção 2) e o objetivo da otimização (A7) usam o **temporal**.
- **Ressalva:** o temporal tem desvio alto (~0,10–0,14), porque cada bloco tem só 33 a 66 fraudes e os blocos diferem entre si (seção 5).

### Quais diferenças no holdout são reais?

O holdout tem só 52 fraudes. Reamostramos as 45.401 linhas do holdout 1.000 vezes (`random_state=42`) e, **em cada reamostragem, calculamos a PR-AUC de todos os modelos nas mesmas linhas**. A diferença pareada entre dois modelos tem IC bem mais estreito que a comparação de dois ICs separados, porque o "azar" da amostra afeta os dois igualmente.

In [ ]:
from sklearn.metrics import average_precision_score

y_h = scores[names[0]]["y"].to_numpy()
rng = np.random.RandomState(42)
idx = [
    i for i in (rng.randint(0, len(y_h), len(y_h)) for _ in range(1000)) if y_h[i].any()
]
boot = {
    n: np.array(
        [average_precision_score(y_h[i], scores[n]["score"].to_numpy()[i]) for i in idx]
    )
    for n in names
}
ref = max(names, key=lambda n: hold[n])
rows = []
for n in names:
    if n == ref:
        continue
    d = boot[ref] - boot[n]
    rows.append(
        {
            "comparação": f"{ref} − {n}",
            "diferença": hold[ref] - hold[n],
            "IC 95% baixo": np.quantile(d, 0.025),
            "IC 95% alto": np.quantile(d, 0.975),
            f"P({ref} melhor)": np.mean(d > 0),
            "significativa": np.quantile(d, 0.025) > 0,
        }
    )
pd.DataFrame(rows).round(4)

**Leitura.** Nenhuma diferença entre a RF e os modelos fortes é significativa no holdout: contra a logística, +0,026 (IC [−0,028; +0,085]; a RF é melhor em 83% das reamostragens); contra o MLP, +0,040 (IC [−0,022; +0,105]); contra o KNN, +0,043 (IC [−0,007; +0,097]); contra o AdaBoost com SMOTE, +0,063 (IC [−0,004; +0,132]). **Só são significativamente piores que a RF:** a Árvore de Decisão (+0,067, IC [+0,007; +0,131]), o KNN com SMOTE e o AdaBoost balanceado. Com 52 fraudes, o holdout não tem poder para separar modelos que diferem em ~0,03–0,05 de PR-AUC. A escolha do campeão (A8) precisa combinar o holdout com o temporal, a estabilidade e o custo, e o teste final no `stream.csv` (75 fraudes) é a confirmação independente.

## 5. Estabilidade fold a fold (estratificado e temporal)

In [ ]:
best = {}  # melhor variante de cada família, escolhida pelo 5-fold TEMPORAL (não pelo holdout)
for n in names:
    fam = family(n)
    if fam not in best or tmean[n] > tmean[best[fam]]:
        best[fam] = n
print("Variante escolhida por família (pelo 5-fold temporal):", best)

fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
for ax, cv_folds, title in [
    (axes[0], folds, "5-fold estratificado"),
    (axes[1], tfolds, "5-fold temporal (TimeSeriesSplit)"),
]:
    for fam, n in best.items():
        f = cv_folds[n]
        ax.plot(
            f.fold,
            f.valid_pr_auc,
            "o-",
            color=COLOR[fam],
            lw=2,
            ms=6,
            label=FAMILY[fam],
        )
    ax.set_xticks(range(1, 6))
    ax.set_xlabel("Fold / split")
    ax.set_title(title)
axes[0].set_ylabel("PR-AUC de validação")
axes[0].legend(
    frameon=False, fontsize=8, ncol=6, loc="upper left", bbox_to_anchor=(0.0, -0.18)
)
fig.suptitle("PR-AUC por fold (melhor variante de cada família)")
fig.savefig(FIGS / "cmp_02_folds.png")

ranks = pd.DataFrame({n: tfolds[n]["valid_pr_auc"].values for n in names}).rank(
    axis=1, ascending=False
)
ranks.index = [f"split temporal {i}" for i in range(1, 6)]
ranks.astype(int)

**Leitura.**
- **Estratificado (esquerda):** as linhas são quase paralelas e a ordem dos modelos se mantém fold a fold (RF e MLP sempre no topo). É a estabilidade que o desvio pequeno indicava.
- **Temporal (direita):** todos os modelos sobem e descem juntos. O **split 3** (`Time` 63.322–76.231 s, 33 fraudes) é difícil para todos (0,49 a 0,63, sem contar o AdaBoost balanceado): as fraudes daquele bloco se parecem menos com as do passado. A variação vem mais **do período** que do modelo.
- **A ordem muda com o tempo:** a logística fica entre as últimas (8ª de 9) nos três primeiros splits e é a **primeira no split 5** (0,867), o mais próximo do holdout. Isso ajuda a explicar por que ela fica tão perto da RF no holdout: no período final, a fronteira simples generaliza tão bem quanto os modelos flexíveis. A RF está entre os 4 primeiros em todos os splits (1º em dois deles); é o modelo mais consistente.

## 6. Curvas precisão × recall no holdout temporal

In [ ]:
fig, ax = plt.subplots(figsize=(6.4, 4.8))
for fam, n in best.items():
    p, r, _ = precision_recall_curve(scores[n]["y"], scores[n]["score"])
    ax.plot(
        r,
        p,
        color=COLOR[fam],
        lw=2,
        label=f"{FAMILY[fam]} ({n}) — PR-AUC {hold[n]:.3f}",
    )
prev = scores[names[0]]["y"].mean()
ax.axhline(
    prev, color="#52514e", ls="--", lw=1, label=f"Aleatório = prevalência ({prev:.4f})"
)
ax.set_xlabel("Recall (fraude)")
ax.set_ylabel("Precisão (fraude)")
ax.set_xlim(0, 1.01)
ax.set_ylim(0, 1.02)
ax.set_title("Curvas PR no holdout temporal (52 fraudes em 45.401)")
ax.legend(frameon=False, fontsize=7.5, loc="lower left")
fig.savefig(FIGS / "cmp_03_curvas_pr.png")

**Leitura.**
- Todas as curvas têm o mesmo formato: precisão alta (> 0,8) até ~65–75% de recall e **queda brusca entre 75% e 80%**. As ~10 a 14 fraudes restantes (20–25% das 52) recebem scores baixos em todos os modelos (score < 0,1 para 13 fraudes na RF, 14 no MLP e 10 no KNN): para as variáveis disponíveis, elas se parecem com transações legítimas. É o teto de recall com precisão útil nesta base.
- **A RF** mantém precisão de ~0,95 até ~75% de recall, a faixa mais útil para operação (poucos falsos alarmes), e por isso tem a maior área.
- **O MLP** tem precisão 1,0 até ~52% de recall (as fraudes mais óbvias), mas cai mais cedo.
- **A logística** perde precisão antes (~0,8 aos 60% de recall), mas tem a cauda mais longa: alcança 85–90% de recall com precisão baixa, o que explica o recall de 0,90 no limiar 0,5.
- **O AdaBoost com SMOTE** perde precisão no meio da curva (~0,7–0,85 entre 40% e 65% de recall).

## 7. Matrizes de confusão no holdout (limiar 0,5)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(9.5, 6))
for ax, (fam, n) in zip(axes.ravel(), best.items()):
    s = scores[n]
    cm = confusion_matrix(s["y"], s["score"] >= 0.5)
    ax.imshow(
        cm, cmap="Blues", norm=plt.matplotlib.colors.LogNorm(vmin=1, vmax=cm.max())
    )
    for (i, j), v in np.ndenumerate(cm):
        ax.text(
            j,
            i,
            f"{v:,}".replace(",", "."),
            ha="center",
            va="center",
            color="#fcfcfb" if v > 3000 else "#0b0b0b",
            fontsize=10,
        )
    ax.set_xticks([0, 1], ["Legítima", "Fraude"])
    ax.set_yticks([0, 1], ["Legítima", "Fraude"])
    ax.set_title(
        f"{FAMILY[fam]}\nrecall {cm[1, 1] / cm[1].sum():.2f} · precisão {cm[1, 1] / max(cm[:, 1].sum(), 1):.2f}",
        fontsize=9,
    )
    ax.grid(False)
for ax in axes[:, 0]:
    ax.set_ylabel("Real")
for ax in axes[1]:
    ax.set_xlabel("Previsto")
fig.suptitle("Matrizes de confusão no holdout temporal (limiar 0,5)")
fig.tight_layout()
fig.savefig(FIGS / "cmp_04_confusao.png")

**Leitura** (holdout, 52 fraudes; o limiar 0,5 é provisório):
- **Regressão logística:** 47 fraudes detectadas, 5 perdidas e **2.076 falsos alarmes**: inviável para operação. Os pesos de classe deslocam os scores, e o limiar certo fica perto de 1 (notebook 02).
- **Árvore:** 41 detectadas, 395 falsos alarmes.
- **Random Forest:** 35 detectadas, 17 perdidas e **só 2 falsos alarmes**. É o oposto da logística: conservadora no limiar 0,5.
- **AdaBoost com SMOTE:** 41 detectadas, 97 falsos alarmes.
- **MLP:** 38 detectadas, 25 falsos alarmes. **KNN:** 38 detectadas, 19 falsos alarmes.
- O limiar 0,5 não é neutro: cada modelo produz scores em outra escala. A comparação justa é a curva PR (seção anterior). Na A8, o limiar do campeão é escolhido na curva PR do holdout, e a matriz de confusão é refeita com ele.

## 8. Treino × validação (overfitting e underfitting)

In [ ]:
diag = pd.DataFrame(
    {
        "CV treino": m("cv_train_pr_auc_mean"),
        "CV validação": m("cv_valid_pr_auc_mean"),
        "CV temporal validação": m("cv_temporal_valid_pr_auc_mean"),
        "holdout: treino (dev)": m("train_pr_auc"),
        "holdout: validação": m("pr_auc"),
    }
)
diag["gap CV (treino − val.)"] = diag["CV treino"] - diag["CV validação"]
diag["gap holdout"] = diag["holdout: treino (dev)"] - diag["holdout: validação"]

fig, ax = plt.subplots(figsize=(7.5, 4.6))
for y_, n in zip(ypos, names):
    tr, va, tv = (
        diag.loc[n, "CV treino"],
        diag.loc[n, "CV validação"],
        diag.loc[n, "CV temporal validação"],
    )
    ax.plot([min(va, tv), tr], [y_, y_], color="#c3c2b7", lw=2, zorder=1)
    ax.scatter(
        tr, y_, s=60, color="#2a78d6", zorder=2, edgecolors="#fcfcfb", linewidths=1.5
    )
    ax.scatter(
        va, y_, s=60, color="#eb6834", zorder=2, edgecolors="#fcfcfb", linewidths=1.5
    )
    ax.scatter(
        tv,
        y_,
        s=60,
        marker="s",
        color="#0b0b0b",
        zorder=2,
        edgecolors="#fcfcfb",
        linewidths=1.5,
    )
ax.scatter(
    [], [], s=60, color="#2a78d6", label="Treino (média dos folds estratificados)"
)
ax.scatter([], [], s=60, color="#eb6834", label="Validação, 5-fold estratificado")
ax.scatter(
    [], [], s=60, marker="s", color="#0b0b0b", label="Validação, 5-fold temporal"
)
ax.set_yticks(ypos, names)
ax.set_xlabel("PR-AUC (desenvolvimento)")
ax.set_title("Treino × validação: distância = sinal de overfitting")
ax.grid(axis="y", visible=False)
ax.legend(frameon=False, fontsize=8, loc="lower left")
fig.savefig(FIGS / "cmp_05_treino_validacao.png")
diag.round(4)

**Leitura** (azul = treino; laranja = validação no estratificado; quadrado = validação no temporal):
- **Overfitting:** RF, MLP e MLP com SMOTE chegam a ~1,0 no treino e caem para 0,73–0,77 no temporal (diferença de 0,23 a 0,27); a Árvore cai de 0,88 para 0,66; o KNN, de 0,92 para 0,71. A distância treino × validação é bem maior quando a validação é temporal. Esses modelos memorizam, e só parte dessa memorização se transfere para o futuro.
- **Underfitting:** a logística tem treino ≈ validação (0,721 × 0,720 × 0,716). Não há overfitting, mas o teto é baixo: o modelo linear não passa de ~0,72 nem no treino.
- **AdaBoost com SMOTE:** a menor distância entre os modelos flexíveis (treino 0,81, temporal 0,77). Os stumps limitam a capacidade.
- **AdaBoost balanceado:** treino e validação baixos (0,28), um underfitting causado pela parada precoce do boosting.
- **Leitura conjunta:** apesar do overfitting, a RF continua tendo a melhor validação temporal (0,774) e o melhor holdout. O excesso de capacidade não a prejudica no futuro; só infla o k-fold estratificado. Regularizar (folhas maiores, profundidade menor) é o que a A7 testa.

## 9. Custo × desempenho

In [ ]:
OFF = {
    "knn-under": (6, -12),
    "mlp": (-7, 5),
    "decision-tree": (-7, -12),
    "mlp-smote": (7, -9),
}  # rótulos sem colisão
fig, ax = plt.subplots(figsize=(6.4, 4.2))
for n in names:
    ax.scatter(
        m("fit_seconds")[n],
        hold[n],
        s=70,
        color=COLOR[family(n)],
        alpha=0.55 if "smote" in n else 1.0,
        edgecolors="#fcfcfb",
        linewidths=1.5,
    )
    ax.annotate(
        n,
        (m("fit_seconds")[n], hold[n]),
        xytext=OFF.get(n, (6, 2)),
        textcoords="offset points",
        fontsize=8,
        ha="right" if OFF.get(n, (6, 2))[0] < 0 else "left",
    )
ax.set_xscale("log")
ax.set_xlabel("Tempo de treino no desenvolvimento (s, escala log)")
ax.set_ylabel("PR-AUC no holdout temporal")
ax.set_title("Custo de treino × PR-AUC")
fig.savefig(FIGS / "cmp_06_custo.png")

**Leitura.** No canto de "barato e bom" ficam a logística (0,3 s de treino, 0,733) e o KNN (0,2 s de treino, mas a predição mais cara). RF, MLP e Árvore custam de 5 a 15 s de treino e têm PR-AUC parecida, com a RF no topo. O AdaBoost com SMOTE custa uma ordem de grandeza a mais (~199 s) sem ganhar no holdout. Para a API, o que importa é o custo de **predição**: RF e MLP pontuam mil transações em ~2 ms e ~0,7 ms, folga enorme para o simulador (2 transações por segundo); o KNN (~27 ms por mil) também caberia, mas cresce com o tamanho do treino.

## 10. Análise crítica e finalistas da A7

**Síntese da A6.**
1. **Melhor modelo:** a **Random Forest** tem a maior PR-AUC no holdout (0,759, IC [0,619; 0,870]), no 5-fold temporal (0,774) e no estratificado (0,854), e está entre os 4 primeiros em todos os splits temporais. A vantagem sobre a logística, o MLP e o KNN **não é estatisticamente significativa** no holdout (52 fraudes), mas é consistente nas três estimativas.
2. **O baseline é competitivo no futuro:** a logística tem PR-AUC quase igual no holdout (0,733), é a mais barata e a mais estável entre treino e validação, mas precisa de um limiar muito alto e tem precisão menor na faixa útil da curva PR.
3. **Validação:** o k-fold estratificado embaralhado superestima os modelos flexíveis em 0,03–0,12; o 5-fold temporal concorda com o holdout. Daqui em diante, o temporal é a referência no desenvolvimento.
4. **Desbalanceamento:** `class_weight` onde existe; SMOTE só no AdaBoost (único caso em que ajudou); KNN com subamostra 1:50; MLP sem SMOTE.
5. **Trade-offs:** a RF no limiar 0,5 é conservadora (precisão 0,95, recall 0,67), e a logística, agressiva (recall 0,90, precisão 0,02). O limiar de operação será escolhido na A8.

**Finalistas da A7:** **Random Forest** (melhor em todas as estimativas) e **MLP** (2º no holdout entre os flexíveis, 3º no temporal). O AdaBoost com SMOTE ficou em 2º no temporal (0,771, empatado com a RF dentro do ruído), mas foi deixado de fora: o treino custa ~13 vezes o da RF e ~17 vezes o do MLP (cada trial da otimização seriam ~5 treinos de 1,5 a 3 min), e o holdout dele é o menor entre os modelos fortes (0,696). A logística não entra na otimização: tem pouco a ajustar (só a regularização) e já serve de referência.